In [1]:
import ee
import os
import glob
import duckdb
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

ee.Authenticate()

# Authenticate & initialize Earth Engine
ee.Initialize(project='biodiversity-478015')

In [2]:
# -------------------------
# CONFIG
# -------------------------
current_dir = os.getcwd()
aoifile = os.path.join(current_dir, 'mav_counties_4326.parquet')
scale = 100  # Adjust based on your imagery resolution
aoi_gdf = gpd.read_parquet(aoifile)
# 1. Union all geometries
unified_geom = aoi_gdf.geometry.union_all()

# 2. Convert Multipolygon to a single Polygon (e.g., using convex hull)
# If your shapes are far apart, consider using .envelope instead
single_geom = unified_geom.convex_hull 

# 3. Create EE Geometry
#aoi_geom = ee.Geometry.Polygon(list(single_geom.exterior.coords))

'''
Spring: March 1 – May 31
Summer: June 1 – August 31
Fall (Autumn): September 1 – November 30
Winter: December 1 – February 28 (or 29 in a leap year) 
'''

'\nSpring: March 1 – May 31\nSummer: June 1 – August 31\nFall (Autumn): September 1 – November 30\nWinter: December 1 – February 28 (or 29 in a leap year) \n'

In [3]:
def gdf_to_ee_geometry(gdf):
    if gdf.empty:
        raise ValueError("GeoDataFrame is empty.")

    # Merge all geometries into one
    geom = gdf.union_all()
    if geom.is_empty:
        raise ValueError("Geometry is empty.")

    # Convert to GeoJSON and then EE Geometry
    geom_json = geom.__geo_interface__
    return ee.Geometry(geom_json)

def get_season_date_range(year, season):
    """Return (start_date, end_date) as ee.Date for the given season and base year.
    Seasons:
      Spring: Mar 1 – May 31
      Summer: Jun 1 – Aug 31
      Fall:   Sep 1 – Nov 30
      Winter: Dec 1 – Feb 28 (crosses into next year)
    end_date is exclusive for filterDate(start, end).
    """
    season = season.lower()
    if season == "spring":
        start = ee.Date.fromYMD(year, 3, 1)
        end   = ee.Date.fromYMD(year, 6, 1)  # exclusive
    elif season == "summer":
        start = ee.Date.fromYMD(year, 6, 1)
        end   = ee.Date.fromYMD(year, 9, 1)
    elif season == "fall":
        start = ee.Date.fromYMD(year, 9, 1)
        end   = ee.Date.fromYMD(year, 12, 1)
    elif season == "winter":
        start = ee.Date.fromYMD(year, 1, 1)
        end   = ee.Date.fromYMD(year+1, 1, 1)
    else:
        raise ValueError(f"Unknown season: {season}")
    return start, end

def get_daymet_season_mean_image(season, year, bands=None):
    """Build a seasonal mean image from DAYMET V4 over [start, end) dates."""
    #start_date, end_date = get_season_date_range(year, season)
    if season in ('spring'):
        ic = ee.ImageCollection("NASA/ORNL/DAYMET_V4").filter(ee.Filter.calendarRange(2017,2024,'year'))
        ic = ic.filter(ee.Filter.calendarRange(3,5,'month'))
    if season in ('summer'):
        ic = ee.ImageCollection("NASA/ORNL/DAYMET_V4").filter(ee.Filter.calendarRange(2017,2024,'year'))
        ic = ic.filter(ee.Filter.calendarRange(6,8,'month'))
    if season in ('fall'):
        ic = ee.ImageCollection("NASA/ORNL/DAYMET_V4").filter(ee.Filter.calendarRange(2017,2024,'year'))
        ic = ic.filter(ee.Filter.calendarRange(9,11,'month'))        
    elif season == 'winter':
        ic = ee.ImageCollection("NASA/ORNL/DAYMET_V4").filter(ee.Filter.calendarRange(2017,2024,'year'))
        ic = ic.filter(ee.Filter.calendarRange(12,2,'month'))
    if bands:
        ic = ic.select(bands)
    # Mean across days in the season
    return ic.mean()  # daily -> seasonal mean

def compute_neighborhood_means(img, radius_m):
    """Return per-band neighborhood means with a circular kernel in meters,
    suffixed with _mean_<radius>m.
    """
    kernel = ee.Kernel.circle(radius_m, 'meters', True)
    reduced = img.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=kernel)
    # Rename <band>_mean -> <band>_mean_<radius>m
    names = img.bandNames()
    new_names = names.map(lambda n: ee.String(n).cat("_").cat(ee.Number(radius_m).format()).cat("m"))
    return reduced.rename(new_names)

def build_daymet_composite(img, radii=10000):
    """Concatenate neighborhood means for the given radii."""
    #parts = [compute_daymet_neighborhood_means(img, r) for r in radii]
    return ee.Image.cat(compute_neighborhood_means(img, radii))

def export_feature_rasters_by_season(
    feature,
    year=2025,
    asset_folder='biodiversity_daymet',
    bands=('dayl', 'prcp', 'tmax', 'tmin'),#('temperature','pressure','total_precipitation'),
    scale=1000,            # DAYMET ~1 km native resolution
    max_pixels=1e13
    ):
    """Export per-season composites of DAYMET neighborhood means (1000 m & 10 km)."""
    try:
        aoi_buffered = feature.buffer(5000).bounds()

        seasons = ["winter", "spring", "summer", "fall"]

        for season in seasons:
            # seasonal mean image, clipped to AOI
            season_img = get_daymet_season_mean_image(season, year, bands).clip(aoi_buffered)

            # concat neighborhood mean bands (100 m & 10 km)
            composite = compute_neighborhood_means(season_img, 10000)
            composite = composite.addBands(ee.Image.constant(seasons.index(season)).toFloat().rename('season'))
            composite = composite.clip(feature)

            export_desc = f"{season}"
            task = ee.batch.Export.image.toDrive(
                image=composite,
                description=export_desc,
                folder=asset_folder,
                fileNamePrefix=export_desc,
                region=feature,      # keep your original region
                scale=scale,         # match DAYMET native grid
                maxPixels=max_pixels
            )
            task.start()
            print(f"✔ Export started for {season} composite ({year})")

    except Exception as e:
        print(f"❌ Export failed: {e}")


In [4]:
aoifc = gdf_to_ee_geometry(aoi_gdf)
export_feature_rasters_by_season(aoifc)

✔ Export started for winter composite (2025)
✔ Export started for spring composite (2025)
✔ Export started for summer composite (2025)
✔ Export started for fall composite (2025)
